# 03 — Instance Segmentation dengan YOLO-seg dan SAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/04-segmentation/03_yolo_seg_dan_sam.ipynb)

Dua pendekatan modern untuk instance segmentation:

| | **YOLO-seg** | **SAM** (*Segment Anything Model*) |
|---|---|---|
| Cara kerja | Deteksi + mask sekaligus untuk kelas yang dilatih (80 kelas COCO) | Membuat mask untuk **apa pun** yang ditunjuk lewat *prompt* (titik atau kotak) |
| Tahu kelas? | Ya | Tidak, hanya "objek di sini" |
| Kegunaan | Aplikasi real-time dengan kelas tetap | Anotasi cepat, objek di luar kelas yang dilatih, pipeline gabungan |

Isi notebook:
1. Inference **YOLO11n-seg** dan membedah mask serta poligonnya.
2. **SAM** dengan prompt titik, beberapa titik (positif/negatif), dan kotak.
3. Pipeline **detektor → SAM**: kotak dari YOLO dipakai sebagai prompt SAM, lalu dibandingkan dengan mask YOLO-seg.

> ⚖️ **Lisensi:** `ultralytics` berlisensi **AGPL-3.0** (termasuk bobot YOLO). Bobot asli SAM, SAM 2, dan MobileSAM dirilis dengan lisensi Apache-2.0, tetapi di notebook ini dipakai lewat `ultralytics`.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi dan Import

`ultralytics` belum terpasang di Colab secara default. Gambar contoh `bus.jpg` dan `zidane.jpg` sudah disertakan di dalam paket.

In [ ]:
%pip install -q ultralytics

Import library dan fungsi bantu untuk menampilkan mask sebagai *overlay* berwarna di atas gambar.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO, SAM
from ultralytics.utils import ASSETS

BUS = ASSETS / "bus.jpg"
ZIDANE = ASSETS / "zidane.jpg"
img_bus = np.array(Image.open(BUS).convert("RGB"))
print("bus.jpg:", img_bus.shape)


def overlay(ax, img, masks, alpha=0.5, judul=""):
    # masks: array (N, H, W) boolean; setiap instance diberi warna berbeda
    ax.imshow(img)
    warna = plt.cm.Set1.colors  # warna kontras (merah, hijau, ungu, ...) agar terlihat di objek biru
    lapisan = np.zeros((*img.shape[:2], 4))
    for i, m in enumerate(masks):
        lapisan[m] = (*warna[i % len(warna)], alpha)
    ax.imshow(lapisan)
    for i, m in enumerate(masks):  # garis tepi mask
        ax.contour(m, levels=[0.5], colors=[warna[i % len(warna)]], linewidths=1.5)
    ax.set_title(judul); ax.axis("off")

## 2. YOLO11n-seg

Model `-seg` menghasilkan kotak **dan** mask untuk setiap objek. Parameter `retina_masks=True` membuat mask berukuran sama dengan gambar asli (default-nya mask berukuran input model yang lebih kecil).

Atribut penting di `r.masks`:
- `r.masks.data`: tensor `(N, H, W)` berisi mask biner.
- `r.masks.xy`: daftar **poligon** dalam piksel, satu per objek.
- `r.masks.xyn`: poligon ternormalisasi 0–1, persis format label YOLO-seg (lihat notebook 01).

In [ ]:
seg_model = YOLO("yolo11n-seg.pt")
r = seg_model.predict(BUS, retina_masks=True, verbose=False)[0]

masks_yolo = r.masks.data.cpu().numpy().astype(bool)
print("jumlah objek:", len(r.boxes), "| masks.data:", masks_yolo.shape)
print(f"{'kelas':<8} {'conf':>5} {'luas mask':>10} {'luas kotak':>11} {'rasio':>6} {'titik poligon':>14}")
for i in range(len(r.boxes)):
    x1, y1, x2, y2 = r.boxes.xyxy[i].tolist()
    luas_mask, luas_box = masks_yolo[i].sum(), (x2 - x1) * (y2 - y1)
    print(f"{r.names[int(r.boxes.cls[i])]:<8} {float(r.boxes.conf[i]):>5.2f} {luas_mask:>10,} "
          f"{luas_box:>11,.0f} {luas_mask / luas_box:>6.2f} {len(r.masks.xy[i]):>14}")

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
ax[0].imshow(r.plot()[..., ::-1]); ax[0].set_title("r.plot()"); ax[0].axis("off")
overlay(ax[1], img_bus, masks_yolo, judul="overlay mask (dibuat sendiri)")
plt.tight_layout(); plt.show()

Kolom **rasio** (luas mask / luas kotak) menunjukkan mengapa mask lebih informatif: orang hanya mengisi sebagian kecil kotaknya, sedangkan bus mengisi hampir penuh.

Poligon dari `r.masks.xyn` bisa langsung ditulis sebagai label YOLO-seg. Ini cara cepat membuat *pre-annotation* untuk dataset baru.

In [ ]:
baris_label = []
for k, poli in zip(r.boxes.cls.int().tolist(), r.masks.xyn):
    baris_label.append(f"{k} " + " ".join(f"{v:.4f}" for v in poli.ravel()))
for b in baris_label[:2]:
    print(b[:100], "...")

# Gambar ulang poligon piksel di atas gambar
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(img_bus)
for i, poli in enumerate(r.masks.xy):
    ax.add_patch(patches.Polygon(poli, closed=True, fill=False, edgecolor=plt.cm.tab10(i), lw=2))
ax.set_title("poligon dari r.masks.xy"); ax.axis("off"); plt.show()

## 3. SAM: Segment Anything

SAM tidak mengenal kelas. Ia menerima **prompt** dan mengembalikan mask untuk objek yang ditunjuk:
- **Titik positif** (`label=1`): "objek di sini".
- **Titik negatif** (`label=0`): "bukan bagian ini", untuk mengoreksi mask.
- **Kotak**: "objek di dalam kotak ini".

Kita memakai **MobileSAM** (`mobile_sam.pt`, sekitar 40 MB), versi ringan SAM yang cukup cepat bahkan di CPU. Untuk kualitas lebih tinggi, coba `sam2.1_b.pt` atau `sam2.1_l.pt`.

In [ ]:
sam = SAM("mobile_sam.pt")

# Prompt 1 titik di badan bus
titik = [[400, 380]]
r_titik = sam.predict(BUS, points=titik, labels=[1], verbose=False)[0]

# Prompt 1 titik di orang paling kiri
r_orang = sam.predict(BUS, points=[[140, 600]], labels=[1], verbose=False)[0]

# Prompt kotak di orang kedua dari kiri
kotak = [225, 400, 345, 865]
r_kotak = sam.predict(BUS, bboxes=[kotak], verbose=False)[0]

for nama, rr in [("titik di bus", r_titik), ("titik di orang", r_orang), ("prompt kotak", r_kotak)]:
    print(f"{nama:<15} luas mask: {int(rr.masks.data.sum()):>8,} piksel")

fig, axes = plt.subplots(1, 3, figsize=(15, 7))
for ax, rr, judul in [(axes[0], r_titik, "titik di bus"), (axes[1], r_orang, "titik di orang"),
                      (axes[2], r_kotak, "prompt kotak")]:
    overlay(ax, img_bus, rr.masks.data.cpu().numpy().astype(bool), judul=judul)
axes[0].plot(*titik[0], "g*", ms=18, mec="k")
axes[1].plot(140, 600, "g*", ms=18, mec="k")
axes[2].add_patch(patches.Rectangle(kotak[:2], kotak[2] - kotak[0], kotak[3] - kotak[1], fill=False, ec="lime", lw=2))
plt.tight_layout(); plt.show()

### 3.1 Mengoreksi mask dengan titik negatif

Perhatikan contoh "titik di bus" di atas: titiknya jatuh di **kaca jendela**, sehingga SAM hanya membuat mask jendela itu, bukan seluruh bus. Luasnya jauh lebih kecil daripada mask bus dari YOLO-seg. Begitu pula titik di jaket orang: hasilnya jaket (dan kepala), tanpa kaki. Satu titik memang **ambigu**: apakah maksudnya "jendela", "bus", atau "bus beserta orang di depannya"?

Solusinya: tambahkan beberapa titik positif di bagian lain objek, dan titik **negatif** untuk memberi tahu SAM bagian mana yang **tidak** termasuk. Ini alur kerja tool anotasi interaktif modern.

In [ ]:
# Beberapa titik positif di bus + titik negatif di orang yang berdiri di depannya
pts = [[400, 380], [650, 300], [120, 300], [140, 600]]
lbl = [1, 1, 1, 0]
r_multi = sam.predict(BUS, points=[pts], labels=[lbl], verbose=False)[0]

fig, ax = plt.subplots(figsize=(6, 7))
overlay(ax, img_bus, r_multi.masks.data.cpu().numpy().astype(bool), judul="3 titik positif (hijau) + 1 negatif (merah)")
for (x, y), label in zip(pts, lbl):
    ax.plot(x, y, "*", color="lime" if label else "red", ms=18, mec="k")
plt.show()

## 4. Pipeline Detektor → SAM

Pola yang sering dipakai di industri: detektor (yang tahu **kelas**) menghasilkan kotak, lalu SAM (yang ahli **bentuk**) membuat mask dari kotak tersebut. Hasilnya instance segmentation untuk kelas apa pun yang bisa dideteksi, tanpa perlu melatih model segmentasi.

Kita bandingkan mask hasil pipeline ini dengan mask YOLO-seg memakai **mask IoU** (notebook 01). IoU yang tinggi berarti keduanya sepakat; IoU yang rendah layak diperiksa secara visual.

In [ ]:
def mask_iou(a, b):
    gabungan = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / gabungan if gabungan else 0.0


kotak_yolo = r.boxes.xyxy.cpu().numpy()
t0 = time.time()
r_pipe = sam.predict(BUS, bboxes=kotak_yolo, verbose=False)[0]
waktu_sam = time.time() - t0
masks_sam = r_pipe.masks.data.cpu().numpy().astype(bool)

print(f"SAM untuk {len(kotak_yolo)} kotak: {waktu_sam:.2f} detik\n")
print(f"{'kelas':<8} {'IoU(YOLO-seg, SAM)':>19}")
for i in range(len(kotak_yolo)):
    print(f"{r.names[int(r.boxes.cls[i])]:<8} {mask_iou(masks_yolo[i], masks_sam[i]):>19.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
overlay(ax[0], img_bus, masks_yolo, judul="YOLO11n-seg")
overlay(ax[1], img_bus, masks_sam, judul="kotak YOLO → MobileSAM")
plt.tight_layout(); plt.show()

## 5. Perbandingan Kecepatan

YOLO-seg dirancang untuk kecepatan, sedangkan SAM untuk kualitas mask dan fleksibilitas. Mari ukur waktunya pada gambar kedua. Satu kali *warm-up* dijalankan dulu agar waktu inisialisasi tidak ikut terhitung.

In [ ]:
def ukur(fn, ulang=3):
    fn()  # warm-up
    t0 = time.time()
    for _ in range(ulang):
        hasil = fn()
    return hasil, (time.time() - t0) / ulang * 1000


r_z, ms_yolo = ukur(lambda: seg_model.predict(ZIDANE, retina_masks=True, verbose=False)[0])
kotak_z = r_z.boxes.xyxy.cpu().numpy()
r_zs, ms_sam = ukur(lambda: sam.predict(ZIDANE, bboxes=kotak_z, verbose=False)[0])

print(f"YOLO11n-seg          : {ms_yolo:7.0f} ms ({len(r_z.boxes)} objek)")
print(f"MobileSAM (dari kotak): {ms_sam:7.0f} ms (belum termasuk waktu deteksi)")

img_z = np.array(Image.open(ZIDANE).convert("RGB"))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
overlay(ax[0], img_z, r_z.masks.data.cpu().numpy().astype(bool), judul=f"YOLO11n-seg ({ms_yolo:.0f} ms)")
overlay(ax[1], img_z, r_zs.masks.data.cpu().numpy().astype(bool), judul=f"kotak → MobileSAM (+{ms_sam:.0f} ms)")
plt.tight_layout(); plt.show()

## Ringkasan

- **YOLO-seg**: kotak + mask + kelas dalam satu model cepat; `r.masks.xyn` langsung berformat label YOLO-seg.
- **SAM**: mask untuk objek apa pun yang ditunjuk lewat titik (positif/negatif) atau kotak, tanpa mengenal kelas.
- **Detektor → SAM**: menggabungkan pengetahuan kelas dari detektor dengan kualitas mask dari SAM, dan berguna untuk *pre-annotation* dataset.
- Pilih berdasarkan kebutuhan: kecepatan dan kelas tetap (YOLO-seg), atau fleksibilitas dan kualitas (SAM).

## Latihan

1. **Anotasi semi-otomatis.** Pilih 5 foto milikmu. Jalankan pipeline YOLO → SAM dan simpan hasilnya sebagai file label YOLO-seg (kelas dari YOLO, poligon dari mask SAM; petunjuk: `cv2.findContours` + normalisasi). Visualisasikan kembali label yang tersimpan untuk memastikan formatnya benar.
2. **Ukuran model SAM.** Bandingkan `mobile_sam.pt`, `sam2.1_t.pt`, dan `sam2.1_b.pt` pada prompt kotak yang sama (misalnya orang di `bus.jpg`): waktu per gambar dan kualitas tepi mask secara visual. Hitung juga IoU antar-model. Model mana yang paling seimbang untuk anotasi di Colab GPU?
3. **Menghitung luas.** Dengan YOLO11n-seg, tulis fungsi yang menghitung **persentase area gambar** yang tertutup setiap kelas (misalnya `person: 12.3%`, `bus: 41.0%`). Tangani mask yang saling tumpang tindih agar piksel tidak terhitung dua kali. Uji pada 3 gambar berbeda.